This is Roberta embedded. It used processed data 

Runtime 3118.4412 + 830.6684

Tech Stack: 
GPU: NVIDIA A100-SXM4-80GB
VRAM (GB): 85.167243264

In [12]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

In [13]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM (GB): 85.167243264


In [39]:
DEV_PROCESSED  = "/content/development_processed.csv"
EVAL_PROCESSED = "/content/evaluation_processed.csv"

df_dev  = pd.read_csv(DEV_PROCESSED)
df_eval = pd.read_csv(EVAL_PROCESSED)

print(df_dev.shape, df_eval.shape)

(79997, 12) (20000, 11)


In [40]:
def build_transformer_text(df):
    return (
        df["title"].astype(str) +
        "\n\n" +
        df["article"].astype(str)
    )

df_dev["tr_text"]  = build_transformer_text(df_dev)
df_eval["tr_text"] = build_transformer_text(df_eval)


In [41]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df_dev,
    test_size=0.15,
    stratify=df_dev["label"],
    random_state=42
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)


Train: (67997, 13)
Val: (12000, 13)


In [42]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.texts  = df["tr_text"].tolist()
        self.labels = df["label"].values if "label" in df else None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item


In [43]:
from transformers import RobertaTokenizer

MODEL_NAME = "roberta-large"
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

In [44]:
train_ds = NewsDataset(train_df, tokenizer)
val_ds   = NewsDataset(val_df, tokenizer)
eval_ds  = NewsDataset(df_eval, tokenizer)


In [45]:
from transformers import RobertaForSequenceClassification

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=7
).cuda()


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [46]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./roberta_out",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",

    logging_steps=100,
    report_to="none",
    save_total_limit=2
)


In [47]:
from transformers import Trainer, TrainingArguments


In [48]:
from sklearn.metrics import f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro")
    }


In [49]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)


In [50]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.717800,0.665350,0.725971
2,0.583100,0.643493,0.739160
3,0.487000,0.663435,0.751187


TrainOutput(global_step=25500, training_loss=0.6354938982795266, metrics={'train_runtime': 3118.4412, 'train_samples_per_second': 65.414, 'train_steps_per_second': 8.177, 'total_flos': 1.9010882237094605e+17, 'train_loss': 0.6354938982795266, 'epoch': 3.0})

In [51]:
load_best_model_at_end=True
metric_for_best_model="macro_f1"


In [52]:
full_train_df = df_dev.copy()

full_train_ds = NewsDataset(
    full_train_df,
    tokenizer,
    max_length=512
)


In [54]:
from transformers import TrainingArguments

args_full = TrainingArguments(
    output_dir="./roberta_full_out",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    fp16=True,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=100,
    report_to="none"
)


In [56]:
trainer_full = Trainer(
    model=trainer.model,
    args=args_full,
    train_dataset=full_train_ds
)




In [57]:
trainer_full.train()

Step,Training Loss
100,0.500300
200,0.499100
300,0.494100
400,0.525200
500,0.508600
600,0.520300
700,0.526400
800,0.496300
900,0.505300
1000,0.529300


TrainOutput(global_step=5000, training_loss=0.4615698677062988, metrics={'train_runtime': 1830.6684, 'train_samples_per_second': 87.396, 'train_steps_per_second': 2.731, 'total_flos': 1.4910594548983603e+17, 'train_loss': 0.4615698677062988, 'epoch': 2.0})

In [58]:
preds = trainer_full.predict(eval_ds)
logits = preds.predictions
final_pred = logits.argmax(axis=1)


In [59]:
submission = pd.DataFrame({
    "Id": df_eval["Id"].astype(int),
    "Predicted": final_pred.astype(int)
})

submission.to_csv("submission_roberta_full.csv", index=False)
